In [ ]:
import sys, os, glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cmdstanpy import from_csv
from utils.generate_data import generate_Friedman_data
from utils.sparsity import forward_pass_tanh
from utils.sparsity_deep import forward_pass_tanh_deep, build_posterior_mask

## Cell 1 — Paths and loader

All results live under `results/regression/single_layer_H16/tanh/friedman/`.
Move completed remote runs there before executing.

In [ ]:
RESULTS_DIR = "results/regression/single_layer_H16/tanh/friedman"
DATA_DIR    = "datasets/friedman"
_KEY = re.compile(r"Friedman_N(\d+)_p\d+_sigma([\d.]+)_seed(\d+)")

# (display label) -> (subdir, num_hidden_layers)
ARCH_MAP = {
    "DHS H16 L1": ("dirichlet_horseshoe_tanh_nodewise_H16_L1", 1),
    "DHS H16 L2": ("dirichlet_horseshoe_tanh_nodewise_H16_L2", 2),
    "DHS H16 L3": ("dirichlet_horseshoe_tanh_nodewise_H16_L3", 3),
    "DHS H32 L1": ("dirichlet_horseshoe_tanh_nodewise_H32_L1", 1),
    "DHS H32 L2": ("dirichlet_horseshoe_tanh_nodewise_H32_L2", 2),
    "DHS H32 L3": ("dirichlet_horseshoe_tanh_nodewise_H32_L3", 3),
    "DHS H64 L1": ("dirichlet_horseshoe_tanh_nodewise_H64_L1", 1),
    "RHS H16 L1": ("regularized_horseshoe_tanh_H16_L1",        1),
    "RHS H16 L2": ("regularized_horseshoe_tanh_H16_L2",        2),
    "Gauss H16 L1": ("gaussian_tanh_H16_L1",                   1),
    "Gauss H16 L2": ("gaussian_tanh_H16_L2",                   2),
    "Gauss H16 L3": ("gaussian_tanh_H16_L3",                   3),
    "Gauss H32 L1": ("gaussian_tanh_H32_L1",                   1),
}

configs = sorted(f.replace(".npz", "") for f in os.listdir(DATA_DIR) if f.endswith(".npz"))

def load_fit(subdir, config):
    files = sorted(glob.glob(os.path.join(RESULTS_DIR, subdir, config, "chain_*.csv")))
    return from_csv(files, method="sample") if files else None

fits = {}
for label, (subdir, _) in ARCH_MAP.items():
    fits[label] = {c: f for c in configs if (f := load_fit(subdir, c)) is not None}
    print(f"{label:16s}: {len(fits[label])} configs loaded")

## Cell 2 — Effective per-weight scale heatmaps: RHS λ vs DHS λ_node · φ

**RHS**: has one independent local scale `lambda[h, p]` per weight — no coupling within a node.

**DHS**: has a node-level scale `lambda_node[h]` and a Dirichlet allocation `phi_data[h, p]`
(sums to 1 per node). The effective per-weight scale is their product.
Weights within the same node share a variance budget — this is the within-node coupling
the Dirichlet adds on top of the global-local structure.

Side-by-side heatmaps of the posterior-mean effective scale `(H, P)` make the structural
difference visible: RHS should show fine-grained independent variation; DHS should show
node-level blocks with concentrated allocation within active nodes.

In [ ]:
TARGET_N   = 200
target_configs = [c for c in configs if f"_N{TARGET_N}_" in c]

# Use one representative seed for the heatmap
TARGET_CONFIG = target_configs[0]

fit_rhs = fits["RHS H16 L1"].get(TARGET_CONFIG)
fit_dhs = fits["DHS H16 L1"].get(TARGET_CONFIG)

# RHS: lambda shape (S, H, P) — independent per-weight local scales
lam_rhs  = fit_rhs.stan_variable("lambda")       # (S, H, P)
eff_rhs  = lam_rhs.mean(axis=0)                  # (H, P)

# DHS: lambda_node (S, H) x phi_data (S, H, P) -> effective scale
lam_node = fit_dhs.stan_variable("lambda_node")  # (S, H)
phi_data = fit_dhs.stan_variable("phi_data")     # (S, H, P)
eff_dhs  = (lam_node[:, :, None] * phi_data).mean(axis=0)  # (H, P)

# Sort nodes by total scale (descending) so active nodes are at the top
order_rhs = np.argsort(eff_rhs.sum(axis=1))[::-1]
order_dhs = np.argsort(eff_dhs.sum(axis=1))[::-1]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, eff, order, title in zip(
        axes,
        [eff_rhs,  eff_dhs],
        [order_rhs, order_dhs],
        [r"RHS: independent $\lambda_{h,p}$",
         r"DHS: $\lambda_{{\rm node},h} \cdot \phi_{h,p}$"]):
    im = ax.imshow(eff[order], aspect="auto", cmap="viridis")
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Input feature $p$", fontsize=12)
    ax.set_ylabel("Node (sorted by total scale)", fontsize=12)
    ax.set_xticks(range(eff.shape[1]))
    ax.set_xticklabels([f"$x_{{{i+1}}}$" for i in range(eff.shape[1])], fontsize=9)
    ax.set_yticks([])
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle(f"Posterior-mean effective per-weight scale  —  {TARGET_CONFIG}", fontsize=13)
plt.tight_layout()
plt.show()

## Cell 3 — Within-node concentration: CV of effective scale (RHS vs DHS)

For each node, compute the coefficient of variation (CV = std/mean) of the effective
per-weight scale across the P input features. A high CV means the node concentrates
its budget on few inputs — this is the Dirichlet-induced within-node sparsity.

Aggregated over all N=200 seeds: distribution of per-node CV for RHS vs DHS.
If the Dirichlet mechanism works, DHS should show systematically higher CV.

In [ ]:
def node_cv(eff):  # eff: (H, P) -> (H,)
    return eff.std(axis=1) / (eff.mean(axis=1) + 1e-12)

cv_data = {"RHS H16 L1": [], "DHS H16 L1": []}
for config in target_configs:
    for label in cv_data:
        fit = fits[label].get(config)
        if fit is None:
            continue
        if "RHS" in label:
            eff = fit.stan_variable("lambda").mean(axis=0)             # (H, P)
        else:
            lam_n = fit.stan_variable("lambda_node")                   # (S, H)
            phi   = fit.stan_variable("phi_data")                      # (S, H, P)
            eff   = (lam_n[:, :, None] * phi).mean(axis=0)             # (H, P)
        cv_data[label].extend(node_cv(eff).tolist())

fig, ax = plt.subplots(figsize=(5, 4))
bp = ax.boxplot(
    [cv_data["RHS H16 L1"], cv_data["DHS H16 L1"]],
    labels=["RHS", "DHS"],
    patch_artist=True,
    showfliers=False,
    medianprops=dict(color="black", lw=2),
)
bp["boxes"][0].set_facecolor("C1"); bp["boxes"][0].set_alpha(0.6)
bp["boxes"][1].set_facecolor("C2"); bp["boxes"][1].set_alpha(0.6)
ax.set_ylabel("Within-node CV of effective scale", fontsize=12)
ax.set_title(f"N={TARGET_N}: DHS concentrates per-node budget\n"
             "(higher CV = more within-node concentration)", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## Cell 4 — Posterior-prune RMSE: RHS vs DHS (and width variants)

Head-to-head sparsity-robustness comparison under posterior pruning of W1.
Also includes DHS H32/H64 to show whether the advantage grows with width.
Addresses R1: posterior-prune is the relevant scheme; show it systematically across N.

In [ ]:
COMPARE_LABELS  = ["Gauss H16 L1", "RHS H16 L1", "DHS H16 L1", "DHS H32 L1", "DHS H64 L1"]
SPARSITY_LEVELS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
N_TEST = 2000

COLORS = {
    "Gauss H16 L1": "C0",
    "RHS H16 L1":   "C1",
    "DHS H16 L1":   "C2",
    "DHS H32 L1":   "C3",
    "DHS H64 L1":   "C4",
}
MARKERS = {
    "Gauss H16 L1": "s",
    "RHS H16 L1":   "^",
    "DHS H16 L1":   "o",
    "DHS H32 L1":   "o",
    "DHS H64 L1":   "o",
}
LINESTYLES = {
    "Gauss H16 L1": "-",
    "RHS H16 L1":   "-",
    "DHS H16 L1":   "-",
    "DHS H32 L1":   "--",
    "DHS H64 L1":   ":",
}

def predict_masked(fit, n_layers, mask_W1, X_test):
    W1  = fit.stan_variable("W_1")          # (S, P, H)
    W_L = fit.stan_variable("W_L")          # (S, H, O)
    b_h = fit.stan_variable("hidden_bias")  # (S, n_layers, H)
    b_o = fit.stan_variable("output_bias")  # (S, O)
    W_int_all = fit.stan_variable("W_internal") if n_layers > 1 else None
    S = W1.shape[0]
    y_hats = np.zeros((S, X_test.shape[0]))
    for s in range(S):
        w1s = W1[s] * mask_W1
        if n_layers == 1:
            y_hats[s] = forward_pass_tanh(
                X_test, w1s, b_h[s, 0], W_L[s], b_o[s].reshape(-1)
            ).squeeze()
        else:
            W_internals = [W_int_all[s, k, :, :] for k in range(W_int_all.shape[1])]
            y_hats[s] = forward_pass_tanh_deep(
                X_test, w1s, W_internals, W_L[s], b_h[s], b_o[s].reshape(-1)
            ).squeeze()
    return y_hats.mean(axis=0)

rows = []
for label in COMPARE_LABELS:
    n_layers = ARCH_MAP[label][1]
    for config, fit in fits[label].items():
        m = _KEY.match(config)
        N, sigma, seed = int(m.group(1)), float(m.group(2)), int(m.group(3))
        _, _, y_tr, _        = generate_Friedman_data(N=N, D=10, sigma=sigma, seed=seed)
        y_mean, y_std        = y_tr.mean(), y_tr.std()
        _, X_te, _, y_te_raw = generate_Friedman_data(N=N_TEST, D=10, sigma=sigma, seed=seed + 999)
        y_te  = (y_te_raw - y_mean) / y_std
        W1_s  = fit.stan_variable("W_1")
        for q in SPARSITY_LEVELS:
            mask   = build_posterior_mask(W1_s, q)
            y_pred = predict_masked(fit, n_layers, mask, X_te)
            rmse   = float(np.sqrt(np.mean((y_pred - y_te) ** 2))) * y_std
            rows.append(dict(label=label, N=N, sparsity=q, rmse=rmse))

df_sp     = pd.DataFrame(rows)
df_sp_agg = (
    df_sp.groupby(["label", "N", "sparsity"])["rmse"]
    .agg(center="mean", spread="std")
    .reset_index()
)

plot_Ns = [100, 200, 500]
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, N in zip(axes, plot_Ns):
    sub = df_sp_agg[df_sp_agg["N"] == N]
    for label in COMPARE_LABELS:
        g = sub[sub["label"] == label].sort_values("sparsity")
        if g.empty:
            continue
        ax.plot(g["sparsity"], g["center"],
                color=COLORS[label], marker=MARKERS[label],
                linestyle=LINESTYLES[label], lw=2.2, label=label)
    ax.set_title(f"N={N}", fontsize=15)
    ax.set_xlabel("Sparsity (W1 pruned)", fontsize=12)
    ax.set_xticks(SPARSITY_LEVELS[::2])
    ax.grid(linestyle="--", alpha=0.4)
    ax.tick_params(labelsize=11)

axes[0].set_ylabel("RMSE (original scale)", fontsize=12)
handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="upper right", frameon=False, fontsize=11)
plt.suptitle("Posterior-prune RMSE: RHS vs DHS (width scaling)", fontsize=13)
plt.tight_layout()
plt.show()

## Cell 5 — Width scaling: baseline RMSE vs H (sparsity = 0)

Directly addresses R2: "I would have expected an analysis of the scaling behaviour
with regards to the BNN width."

For each (model family, H, N), show mean RMSE at sparsity=0.
DHS runs from H=16 to H=64; Gauss from H=16 to H=32; RHS at H=16 only.
Shows whether DHS scales more gracefully than Gaussian as width grows.

In [ ]:
df_base = df_sp_agg[df_sp_agg["sparsity"] == 0.0].copy()
df_base["H"]      = df_base["label"].str.extract(r"H(\d+)").astype(int)
df_base["family"] = df_base["label"].str.split().str[0]   # "DHS", "RHS", "Gauss"

FAMILY_COLORS = {"Gauss": "C0", "RHS": "C1", "DHS": "C2"}
FAMILY_MARKERS = {"Gauss": "s", "RHS": "^", "DHS": "o"}

# Only L1 for a clean width comparison
df_width = df_base[df_base["label"].str.endswith("L1")]

plot_Ns = [100, 200, 500]
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
for ax, N in zip(axes, plot_Ns):
    sub = df_width[df_width["N"] == N]
    for family, grp in sub.groupby("family"):
        g = grp.sort_values("H")
        ax.plot(g["H"], g["center"],
                color=FAMILY_COLORS[family], marker=FAMILY_MARKERS[family],
                lw=2.2, label=family)
    ax.set_title(f"N={N}", fontsize=15)
    ax.set_xlabel("Hidden units H", fontsize=12)
    ax.set_xticks(sorted(df_width["H"].unique()))
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.tick_params(labelsize=11)

axes[0].set_ylabel("RMSE (original scale)", fontsize=12)
handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="upper right", frameon=False, fontsize=12)
plt.suptitle("Width scaling: baseline RMSE vs H (L1, sparsity=0)", fontsize=13)
plt.tight_layout()
plt.show()

## Cell 6 — Depth scaling: RMSE vs depth (L1/L2/L3) for DHS H16 and Gauss H16

Addresses the other half of R2's scaling critique. H is fixed at 16 for a clean
depth comparison. Shows whether DHS maintains or improves its advantage as depth grows.

In [ ]:
DEPTH_LABELS = [
    "Gauss H16 L1", "Gauss H16 L2", "Gauss H16 L3",
    "DHS H16 L1",   "DHS H16 L2",   "DHS H16 L3",
    "RHS H16 L1",   "RHS H16 L2",
]

# Compute RMSE for any labels not yet in df_sp_agg
existing = set(df_sp_agg["label"].unique())
extra_rows = []
for label in DEPTH_LABELS:
    if label in existing:
        continue
    n_layers = ARCH_MAP[label][1]
    for config, fit in fits[label].items():
        m = _KEY.match(config)
        N, sigma, seed = int(m.group(1)), float(m.group(2)), int(m.group(3))
        _, _, y_tr, _        = generate_Friedman_data(N=N, D=10, sigma=sigma, seed=seed)
        y_mean, y_std        = y_tr.mean(), y_tr.std()
        _, X_te, _, y_te_raw = generate_Friedman_data(N=N_TEST, D=10, sigma=sigma, seed=seed + 999)
        y_te  = (y_te_raw - y_mean) / y_std
        W1_s  = fit.stan_variable("W_1")
        mask  = build_posterior_mask(W1_s, 0.0)
        y_pred = predict_masked(fit, n_layers, mask, X_te)
        rmse   = float(np.sqrt(np.mean((y_pred - y_te) ** 2))) * y_std
        extra_rows.append(dict(label=label, N=N, sparsity=0.0, rmse=rmse))

if extra_rows:
    df_extra = pd.DataFrame(extra_rows)
    df_extra_agg = (
        df_extra.groupby(["label", "N", "sparsity"])["rmse"]
        .agg(center="mean", spread="std").reset_index()
    )
    df_all_base = pd.concat([df_sp_agg[df_sp_agg["sparsity"] == 0.0], df_extra_agg], ignore_index=True)
else:
    df_all_base = df_sp_agg[df_sp_agg["sparsity"] == 0.0].copy()

df_all_base["L"]      = df_all_base["label"].str.extract(r"L(\d+)").astype(int)
df_all_base["family"] = df_all_base["label"].str.split().str[0]
df_depth = df_all_base[
    df_all_base["label"].isin(DEPTH_LABELS) & df_all_base["label"].str.contains("H16")
]

plot_Ns = [100, 200, 500]
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
for ax, N in zip(axes, plot_Ns):
    sub = df_depth[df_depth["N"] == N]
    for family, grp in sub.groupby("family"):
        g = grp.sort_values("L")
        ax.plot(g["L"], g["center"],
                color=FAMILY_COLORS[family], marker=FAMILY_MARKERS[family],
                lw=2.2, label=family)
    ax.set_title(f"N={N}", fontsize=15)
    ax.set_xlabel("Depth (num. hidden layers)", fontsize=12)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(["L1", "L2", "L3"], fontsize=12)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.tick_params(labelsize=11)

axes[0].set_ylabel("RMSE (original scale)", fontsize=12)
handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="upper right", frameon=False, fontsize=12)
plt.suptitle("Depth scaling: baseline RMSE vs L (H=16, sparsity=0)", fontsize=13)
plt.tight_layout()
plt.show()